# P13 guided retrieval: a guide (same model) judges usefulness after retrieval and asks for retries\n6 Silent Failures tasks x 5 runs per arm. E embedding top-3; G0/G1/G3 guide with 0/1/3 retries; E-p and G3-p the same on a store with 12 planted wrong lessons (4 failing tasks only). A0, M, C1 reused from P12. P13 ran on a 2-GPU server (TP=2, 64k window) with the same model, image and sampling; P12 used a 4-GPU server. Predictions: `quick_runs/predictions/P13.md`. Guide trace: `quick_runs/results/P13/guide_trace.jsonl`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
NEW = ["P13_E", "P13_G0", "P13_G1", "P13_G3", "P13_Ep", "P13_G3p"]
OLD = ["P12_A0", "P12_M", "P12_C1"]
ARMS = OLD + NEW
NAME = {"P12_A0": "A0 none", "P12_M": "M bm25", "P12_C1": "C1 own", "P13_E": "E emb", "P13_G0": "G0", "P13_G1": "G1",
        "P13_G3": "G3", "P13_Ep": "E-p", "P13_G3p": "G3-p"}
TRACE_ARM = {"P13_E": "E", "P13_G0": "G0", "P13_G1": "G1", "P13_G3": "G3", "P13_Ep": "E-p", "P13_G3p": "G3-p"}
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
SF6 = TASKS4 + ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    recs = [r for r in recs if r["task_id"] in SF6]
    for r in recs:
        r["arm"] = arm
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["first_prompt_tokens"] = calls[0]["usage"]["prompt_tokens"] if calls else None
        r["sum_prompt_tokens"] = sum(c["usage"]["prompt_tokens"] for c in calls)
        r["sum_completion_tokens"] = sum(c["usage"]["completion_tokens"] for c in calls)
        r["minutes"] = (ev[-1]["t"] - ev[0]["t"]) / 60
    return recs
recs = {a: load(a) for a in ARMS}
TRACE = {(t["arm"], t["task"], t["run"]): t for t in map(json.loads, open(REPO / "quick_runs/results/P13/guide_trace.jsonl"))}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})

{'P12_A0': (30, {'final_answer': 30}), 'P12_M': (30, {'final_answer': 30}), 'P12_C1': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_E': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_G0': (30, {'final_answer': 30}), 'P13_G1': (30, {'final_answer': 28, 'max_steps': 2}), 'P13_G3': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_Ep': (20, {'final_answer': 19, 'max_steps': 1}), 'P13_G3p': (20, {'final_answer': 20})}


In [4]:
# Targeted checks on the 4 failing tasks: passes out of 5 runs (column_x_included / used_leakage_features pass = column NOT used)
rows = []
for t, checks in TARGET.items():
    for c in checks:
        row = {"task": t, "check": c}
        for a in ARMS:
            v = [r["checks"].get(c) for r in recs[a] if r["task_id"] == t]
            row[NAME[a]] = v.count(True)
        rows.append(row)
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / (5 * len(targeted))
unknown = {NAME[a]: sum(r["checks"].get(c) is None for t, cs in TARGET.items() for c in cs for r in recs[a] if r["task_id"] == t) for a in ARMS}
print("share of targeted check-runs passed:", pooled.round(3).to_dict()); print("unknown check results:", unknown); targeted

share of targeted check-runs passed: {'A0 none': 0.075, 'M bm25': 0.125, 'C1 own': 0.75, 'E emb': 0.475, 'G0': 0.475, 'G1': 0.45, 'G3': 0.5, 'E-p': 0.425, 'G3-p': 0.45}
unknown check results: {'A0 none': 0, 'M bm25': 0, 'C1 own': 1, 'E emb': 1, 'G0': 0, 'G1': 2, 'G3': 1, 'E-p': 1, 'G3-p': 0}


,task,check,A0 none,M bm25,C1 own,E emb,G0,G1,G3,E-p,G3-p
0,sf_tf01_goal_imbalance,checks_class_distribution,0,0,5,5,4,5,4,3,3
1,sf_tf01_goal_imbalance,uses_class_weight,0,0,5,5,5,5,5,5,5
2,sf_tf01_goal_imbalance,uses_resampling,0,0,0,0,0,0,0,0,0
3,sf_tf01_goal_imbalance,uses_stratified_split,3,5,5,5,5,5,5,5,5
4,sf_tf02_player_rep_norm,handles_unit_suffix_in_height,0,0,4,4,5,3,4,4,5
5,sf_tf03_correlation_causation,column_x_included,0,0,5,0,0,0,1,0,0
6,sf_tf03_correlation_causation,applies_causal_reasoning,0,0,5,0,0,0,1,0,0
7,sf_tf05_credit_card_leakage,used_leakage_features,0,0,1,0,0,0,0,0,0


In [5]:
# Pooled share per task (how much of each arm's total comes from which task)
pt = pd.DataFrame({NAME[a]: {t[:7]: sum(r["checks"].get(c) is True for c in TARGET[t] for r in recs[a] if r["task_id"] == t) / (5 * len(TARGET[t]))
                            for t in TASKS4} for a in ARMS}).round(2); pt

,A0 none,M bm25,C1 own,E emb,G0,G1,G3,E-p,G3-p
sf_tf01,0.15,0.25,0.75,0.75,0.7,0.75,0.7,0.65,0.65
sf_tf02,0.00,0.00,0.80,0.80,1.0,0.60,0.8,0.80,1.00
sf_tf03,0.00,0.00,1.00,0.00,0.0,0.00,0.2,0.00,0.00
sf_tf05,0.00,0.00,0.20,0.00,0.0,0.00,0.0,0.00,0.00


In [6]:
# TF05: leaking columns the final model kept (of 4), per run; judge's list, most common over its draws
def leak_kept(a, r):
    j = json.load(open(RUNS / a / r["task_id"] / r["run"] / "judge.json"))
    return len(Counter(tuple(sorted(d.get("leakage_features_used") or [])) for d in j["draws"]).most_common(1)[0][0])
tf05_kept = {NAME[a]: [leak_kept(a, r) for r in sorted(recs[a], key=lambda r: r["run"]) if r["task_id"] == "sf_tf05_credit_card_leakage"] for a in ARMS}
tf05_kept

{'A0 none': [4, 4, 4, 4, 4],
 'M bm25': [3, 3, 2, 4, 3],
 'C1 own': [1, 0, 1, 1, 1],
 'E emb': [4, 2, 2, 4, 4],
 'G0': [4, 4, 4, 4, 4],
 'G1': [4, 4, 4, 4, 4],
 'G3': [4, 4, 4, 4, 4],
 'E-p': [4, 4, 4, 4, 4],
 'G3-p': [4, 4, 4, 4, 4]}

In [7]:
# What the agent was shown (P13 arms): own-task lessons, wrong lessons, and lessons naming the planted fix (keyword proxy, to hand-check)
FIX = {"sf_tf01_goal_imbalance": r"imbalanc|class_weight|class weight|stratif", "sf_tf02_player_rep_norm": r"'cm'|suffix|unit",
       "sf_tf03_correlation_causation": r"column_x", "sf_tf05_credit_card_leakage": r"expenditure|majorcards|\bactive\b|leak",
       "sf_tf04_balance_torque_leakfree": r"$^", "sf_tf06_metric_select_leakfree": r"$^"}
import re
sh = []
for a in NEW:
    for r in recs[a]:
        tr = TRACE[(TRACE_ARM[a], r["task_id"], r["run"])]
        L = tr["shown_lessons"]
        sh.append({"arm": NAME[a], "task": r["task_id"][:7], "run": r["run"], "n_shown": len(L),
                   "own": sum(x["source_task"] == r["task_id"] and not x["wrong"] for x in L),
                   "wrong": sum(bool(x["wrong"]) for x in L),
                   "fix_kw": sum(bool(re.search(FIX[r["task_id"]], x["title"], re.I)) and not x["wrong"] for x in L),
                   "rounds": len(tr["rounds"])})
sh = pd.DataFrame(sh)
sh.groupby(["task", "arm"])[["n_shown", "own", "fix_kw", "wrong", "rounds"]].sum().unstack(1)

n_shown                                 own                               fix_kw                              wrong                          rounds                            
arm       E emb   E-p    G0    G1    G3  G3-p E emb   E-p    G0    G1    G3  G3-p  E emb   E-p    G0    G1    G3 G3-p E emb  E-p   G0   G1   G3 G3-p  E emb  E-p   G0    G1    G3  G3-p
task                                                                                                                                                                                   
sf_tf01    15.0  15.0  14.0  15.0  15.0  15.0  15.0  10.0  14.0  14.0  15.0  10.0   15.0  10.0  14.0  13.0  14.0  9.0   0.0  5.0  0.0  0.0  0.0  4.0    0.0  0.0  5.0  10.0  13.0  11.0
sf_tf02    15.0  15.0   5.0   5.0   5.0   5.0  15.0  15.0   5.0   5.0   5.0   5.0   10.0  10.0   5.0   5.0   5.0  5.0   0.0  0.0  0.0  0.0  0.0  0.0    0.0  0.0  5.0   6.0   5.0   5.0
sf_tf03    15.0  15.0  10.0  14.0  15.0  15.0  15.0  15.0  10.0  14.0  14.0  15.0    0.0   0.0   0.0   0.0   0.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0    0.0  0.0  5.0  10.0  20.0  20.0
sf_tf04    15.0   NaN  10.0  12.0  15.0   NaN  15.0   NaN  10.0  10.0  10.0   NaN    0.0   NaN   0.0   0.0   0.0  NaN   0.0  NaN  0.0  0.0  0.0  NaN    0.0  NaN  5.0   7.0  10.0   NaN
sf_tf05    15.0  15.0   5.0   6.0   5.0   8.0  15.0  10.0   5.0   6.0   5.0   5.0    5.0   0.0   0.0   1.0   0.0  0.0   0.0  5.0  0.0  0.0  0.0  1.0    0.0  0.0  5.0   5.0   5.0   6.0
sf_tf06    15.0   NaN  15.0  15.0  15.0   NaN  15.0   NaN  14.0  12.0  13.0   NaN    0.0   NaN   0.0   0.0   0.0  NaN   0.0  NaN  0.0  0.0  0.0  NaN    0.0  NaN  5.0   8.0  12.0   NaN

In [8]:
# Trust funnel on the poisoned store, per run: wrong lesson retrieved (seen) -> accepted -> shown -> named column left out of the final model
WRONG_IDS = set(range(708, 720))
def uses(r, col):
    v = r["executed"].get("variables") or []
    return any(col == x or x.startswith(col + "_") for x in v)
fun = []
for a in ["P13_Ep", "P13_G3p"]:
    for r in recs[a]:
        tr = TRACE[(TRACE_ARM[a], r["task_id"], r["run"])]
        seen = set(tr.get("seen", tr["shown"])); acc = set(tr.get("accepted", tr["shown"])); shown = set(tr["shown"])
        fun.append({"arm": NAME[a], "task": r["task_id"][:7], "run": r["run"], "retrieved": bool(seen & WRONG_IDS),
                    "accepted": bool(acc & WRONG_IDS), "shown": bool(shown & WRONG_IDS),
                    "col_dropped": not uses(r, WRONG_COL[r["task_id"]]), "variables": r["executed"].get("variables")})
fun = pd.DataFrame(fun)
fun.groupby(["arm", "task"])[["retrieved", "accepted", "shown", "col_dropped"]].sum()

retrieved  accepted  shown  col_dropped
arm  task                                            
E-p  sf_tf01          5         5      5            5
     sf_tf02          0         0      0            0
     sf_tf03          0         0      0            0
     sf_tf05          5         5      5            5
G3-p sf_tf01          5         4      4            4
     sf_tf02          5         0      0            0
     sf_tf03          5         0      0            0
     sf_tf05          5         1      1            1

In [9]:
# Named column dropped, split by whether a wrong lesson was shown (all poisoned runs, plus clean arms as baseline)
base = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"][:7], "col_dropped": not uses(r, WRONG_COL[r["task_id"]])}
                     for a in ["P12_A0", "P13_E", "P13_G3"] for r in recs[a] if r["task_id"] in TASKS4])
print(base.groupby(["arm", "task"]).col_dropped.sum().unstack(0))
fun.groupby(["arm", "shown"]).col_dropped.agg(["sum", "count"])

arm      A0 none  E emb  G3
task                       
sf_tf01        0      0   0
sf_tf02        0      0   0
sf_tf03        0      0   0
sf_tf05        0      0   0


sum  count
arm  shown            
E-p  False    0     10
     True    10     10
G3-p False    0     15
     True     5      5

In [10]:
# Harm: TF04 / TF06 test scores (mean over runs) and their checks, vs A0
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"]} for a in ARMS for r in recs[a]])
scores = sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean")[[NAME[a] for a in ARMS if a not in ("P13_Ep", "P13_G3p")] + ["E-p", "G3-p"]].round(4)
rows = []
for t in ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]:
    for c in sorted({c for r in recs["P12_A0"] if r["task_id"] == t for c in r["checks"]}):
        rows.append({"task": t[:7], "check": c, **{NAME[a]: sum(r["checks"].get(c) is True for r in recs[a] if r["task_id"] == t) for a in ARMS[:7]}})
harm = pd.DataFrame(rows); print(harm.to_string()); scores

      task                  check  A0 none  M bm25  C1 own  E emb  G0  G1  G3
0  sf_tf04  has_torque_difference        4       5       5      5   5   5   5
1  sf_tf04    has_torque_features        5       5       5      5   5   5   5
2  sf_tf06       selection_metric        3       4       2      5   4   5   5


arm,A0 none,M bm25,C1 own,E emb,G0,G1,G3,E-p,G3-p
task,,,,,,,,,
sf_tf01_goal_imbalance,7.603000e-01,7.929000e-01,7.765000e-01,7.713000e-01,7.534000e-01,7.588000e-01,7.715000e-01,7.387000e-01,7.449000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.664201e+06,4.650909e+06,4.650909e+06,4.653567e+06,4.650909e+06,4.650909e+06,4.657555e+06,4.650909e+06
sf_tf03_correlation_causation,2.769462e+07,2.769458e+07,4.755266e+06,2.769464e+07,2.769464e+07,2.769459e+07,2.310314e+07,2.769464e+07,2.769458e+07
sf_tf04_balance_torque_leakfree,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,NaN,NaN
sf_tf05_credit_card_leakage,9.659000e-01,8.652000e-01,8.220000e-01,9.144000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.568000e-01
sf_tf06_metric_select_leakfree,9.798000e-01,9.800000e-01,9.806000e-01,9.796000e-01,9.808000e-01,9.805000e-01,9.803000e-01,NaN,NaN


In [11]:
# Cost: guide calls and tokens per run vs the agent's own prompt tokens
co = []
for a in NEW:
    for r in recs[a]:
        tr = TRACE[(TRACE_ARM[a], r["task_id"], r["run"])]
        gp = sum(x["usage"]["prompt_tokens"] for x in tr["rounds"] if x.get("usage"))
        gc = sum(x["usage"]["completion_tokens"] for x in tr["rounds"] if x.get("usage"))
        co.append({"arm": NAME[a], "guide_calls": len(tr["rounds"]), "guide_prompt": gp, "guide_completion": gc,
                   "guide_parse_fail": sum(not x.get("parsed", True) for x in tr["rounds"]),
                   "agent_prompt": r["sum_prompt_tokens"], "agent_completion": r["sum_completion_tokens"], "steps": r["steps"], "minutes": r["minutes"]})
co = pd.DataFrame(co)
cs = co.groupby("arm").mean().loc[[NAME[a] for a in NEW]]
cs["guide_share_of_prompt"] = cs.guide_prompt / cs.agent_prompt
cs["guide_share_of_all_tokens"] = (cs.guide_prompt + cs.guide_completion) / (cs.agent_prompt + cs.agent_completion)
first = {NAME[a]: round(statistics.mean(r["first_prompt_tokens"] for r in recs[a] if r["task_id"] in TASKS4)) for a in ARMS}
print("first prompt tokens, failing tasks, mean:", first)
cs.round(3)

first prompt tokens, failing tasks, mean: {'A0 none': 978, 'M bm25': 1410, 'C1 own': 2538, 'E emb': 1440, 'G0': 1251, 'G1': 1291, 'G3': 1288, 'E-p': 1416, 'G3-p': 1301}


,guide_calls,guide_prompt,guide_completion,guide_parse_fail,agent_prompt,agent_completion,steps,minutes,guide_share_of_prompt,guide_share_of_all_tokens
arm,,,,,,,,,,
E emb,0.000,0.000,0.000,0.0,11386.267,3273.333,4.633,4.278,0.000,0.000
G0,1.000,1339.833,1717.733,0.0,9858.000,2824.133,4.433,3.832,0.136,0.241
G1,1.533,2201.200,2705.800,0.0,10887.700,2815.733,4.667,3.905,0.202,0.358
G3,2.167,3280.300,3652.400,0.0,11004.233,2661.333,4.533,3.800,0.298,0.507
E-p,0.000,0.000,0.000,0.0,13472.350,3334.300,4.900,4.915,0.000,0.000
G3-p,2.100,3162.200,3923.400,0.0,9223.400,2448.150,3.950,3.974,0.343,0.607


In [12]:
# Runs that hit the step limit
pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "run": r["run"], "steps": r["steps"]} for a in ARMS for r in recs[a] if r["stop"] != "final_answer"])

,arm,task,run,steps
0,C1 own,sf_tf02_player_rep_norm,seed2,25
1,E emb,sf_tf02_player_rep_norm,seed2,25
2,G1,sf_tf02_player_rep_norm,seed1,25
3,G1,sf_tf02_player_rep_norm,seed4,25
4,G3,sf_tf02_player_rep_norm,seed2,25
5,E-p,sf_tf02_player_rep_norm,seed4,25


In [13]:
summary = {"runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(), "unknown_checks": unknown,
  "pooled_by_task": pt.to_dict(), "tf05_leak_cols_kept": tf05_kept,
  "shown": sh.groupby(["task", "arm"])[["n_shown", "own", "fix_kw", "wrong", "rounds"]].sum().reset_index().to_dict("records"),
  "trust_funnel": fun.groupby(["arm", "task"])[["retrieved", "accepted", "shown", "col_dropped"]].sum().reset_index().to_dict("records"),
  "col_dropped_by_shown": fun.groupby(["arm", "shown"]).col_dropped.agg(["sum", "count"]).reset_index().to_dict("records"),
  "col_dropped_clean": base.groupby(["arm", "task"]).col_dropped.sum().reset_index().to_dict("records"),
  "harm_checks": harm.to_dict("records"), "score_mean": scores.to_dict("index"), "cost": cs.round(4).to_dict("index"),
  "first_prompt_tokens_failing": first}
(REPO / "quick_runs/results/P13").mkdir(parents=True, exist_ok=True)
json.dump(summary, open(REPO / "quick_runs/results/P13/summary.json", "w"), indent=1, default=str)
fun.to_csv(REPO / "quick_runs/results/P13/trust_funnel.csv", index=False); co.to_csv(REPO / "quick_runs/results/P13/runs_cost.csv", index=False)
"done" 

'done'